In [ ]:

# Databricks notebook: Weather API ingestion and update/insert into bronze.climatedata
import requests
import pandas as pd
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.functions import col
# ------------------------------
# CONFIGURATION
# ------------------------------
API_KEY = "0369e5910fbc072b2ce89ea2b4d2deed"  
CITIES = ["New York", "New Delhi" , "London"]
BRONZE_TABLE = "bronze.climatedata"
# ------------------------------
# FETCH DATA FROM WEATHERSTACK API
# ------------------------------
data_list = []
for city in CITIES:
    url = f"https://api.weatherstack.com/current?access_key={API_KEY}&query={city}"
    response = requests.get(url)
    data = response.json()
    if "location" in data and "current" in data:
        row = {
            "country": data["location"].get("country"),
            "location_name": data["location"].get("name"),
            "region": data["location"].get("region"),
            "latitude": data["location"].get("lat"),
            "longitude": data["location"].get("lon"),
            "timezone": data["location"].get("timezone_id"),
            "last_updated_epoch": int(datetime.utcnow().timestamp()),
            "last_updated": datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
            "temperature_celsius": data["current"].get("temperature"),
"temperature_fahrenheit": data["current"].get("temperature") * 9/5 + 32 if                       data["current"].get("temperature") is not None else None,
            "condition_text": data["current"].get("weather_descriptions", [None])[0],
            "wind_mph": data["current"].get("wind_speed"),
            "wind_kph": round(data["current"].get("wind_speed", 0) * 1.60934, 2),
            "wind_degree": data["current"].get("wind_degree"),
            "wind_direction": data["current"].get("wind_dir"),
            "pressure_mb": data["current"].get("pressure"),
            "pressure_in": round(data["current"].get("pressure", 0) * 0.02953, 2),
            "precip_mm": data["current"].get("precip"),
            "precip_in": round(data["current"].get("precip", 0) * 0.03937, 2),
            "humidity": data["current"].get("humidity"),
            "cloud": data["current"].get("cloudcover"),
            "feels_like_celsius": data["current"].get("feelslike"),
            "feels_like_fahrenheit": data["current"].get("feelslike") * 9/5 + 32 if                    
             data["current"].get("feelslike") is not None else None,
            "visibility_km": data["current"].get("visibility"),
            "visibility_miles": round(data["current"].get("visibility", 0) * 0.621371, 2),
            "uv_index": data["current"].get("uv_index"),
            "gust_mph": data["current"].get("gust_mph"),
            "gust_kph": data["current"].get("gust_kph"),
            "air_quality_Carbon_Monoxide": data["current"].get("air_quality_Carbon_Monoxide"),
            "air_quality_Ozone": data["current"].get("air_quality_Ozone"),
            "air_quality_Nitrogen_dioxide": data["current"].get("air_quality_Nitrogen_dioxide"),
            "air_quality_Sulphur_dioxide": data["current"].get("air_quality_Sulphur_dioxide"),
            "air_quality_PM2.5": data["current"].get("air_quality_PM2.5"),
            "air_quality_PM10": data["current"].get("air_quality_PM10"),
            "air_quality_us_epa_index": data["current"].get("air_quality_us_epa_index"),
            "air_quality_gb_defra_index": data["current"].get("air_quality_gb_defra_index"),
            "sunrise": data["current"].get("sunrise"),
            "sunset": data["current"].get("sunset"),
            "moonrise": data["current"].get("moonrise"),
            "moonset": data["current"].get("moonset"),
            "moon_phase": data["current"].get("moon_phase"),
            "moon_illumination": data["current"].get("moon_illumination"),
        }
        data_list.append(row)
# Convert to DataFrame
pdf = pd.DataFrame(data_list)
df = spark.createDataFrame(pdf)
# ------------------------------
# UPSERT TO BRONZE TABLE
# ------------------------------
if DeltaTable.isDeltaTable(spark, f"/user/hive/warehouse/{BRONZE_TABLE.replace('.', '/')}"):
    delta_table = DeltaTable.forName(spark, BRONZE_TABLE)

    delta_table.alias("t").merge(
        source=df.alias("s"),
        condition="t.city = s.location_name AND t.last_updated = s.last_updated",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df.write.format("delta").mode("overwrite").saveAsTable(BRONZE_TABLE)

